In [4]:
import glob
import os
import h5py
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.gridspec as gridspec
import ipywidgets

dataset_names = [
    "Ex", "Ey", "Ez",
    "Bx", "By", "Bz",
    "Jx", "Jy", "Jz",
    "Density"
]

In [9]:
file_list = sorted(glob.glob("data/*.h5"))
if not file_list:
    raise FileNotFoundError("Файлы по шаблону 'data/*.h5' не найдены.")

fig = plt.figure(figsize=(15, 16))
gs = gridspec.GridSpec(4, 3, figure=fig, height_ratios=[1, 1, 1, 1])

axes = []

for row in range(3):
    for col in range(3):
        axes.append(fig.add_subplot(gs[row, col]))

axes.append(fig.add_subplot(gs[3, 1]))

ims = []

with h5py.File(file_list[0], 'r') as f:
    for i, name in enumerate(dataset_names):
        ax = axes[i]
        data = f[name][:]

        cmap = "viridis" if name == "Density" else "RdBu_r"
        vmax = data.max()
        vmin = data.min() if name == "Density" else -abs(vmax)

        im = ax.imshow(data, cmap=cmap, origin="lower", vmin=vmin, vmax=vmax, aspect="auto")
        ax.set_title(name)
        ax.axis("off")

        fig.colorbar(im, ax=ax, orientation="vertical", shrink=0.7)
        ims.append(im)

def update(frame_idx):
    file_path = file_list[frame_idx]
    with h5py.File(file_path, 'r') as f:
        for i, name in enumerate(dataset_names):
            data = f[name][:]
            ims[i].set_data(data)

    fig.suptitle(f"Frame: {os.path.basename(file_path)}", fontsize=16)

    return ims

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(file_list),
    interval=100,
    blit=False
)

output_filename = "datasets_animation.mp4"
writer = animation.FFMpegWriter(fps=3, metadata=dict(artist='Me'), bitrate=1800)

print(f"Начинается запись видео из {len(file_list)} кадров...")
ani.save(output_filename, writer=writer)
print(f"Видео успешно сохранено как {output_filename}")

plt.close(fig)

Начинается запись видео из 2 кадров...
Видео успешно сохранено как datasets_animation.mp4


In [10]:
ipywidgets.Video.from_file(output_filename)

Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')